# Forecasting & Anomaly Detection Experiments (Milestone 5)

This notebook evaluates forecasting models and anomaly detection for the India and Maharashtra groundwater datasets. It strictly enforces chronological splitting and evaluates models against transparent baselines without future leakage.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")
%matplotlib inline

# Project imports
import sys
sys.path.append("../src")
from features.baselines import compute_persistence_baseline, compute_seasonal_historical_baseline, compute_maharashtra_persistence_baseline, compute_baseline_evaluation_metrics
from models.split import chronological_split, print_split_summary
from models.evaluation import evaluate_forecast, compare_models


## 1. Load Data
Load the normalized India and Maharashtra datasets.

In [2]:
DATA_DIR = Path("../data")
INDIA_NORMALIZED = DATA_DIR / "processed/india/india_normalized.parquet"
MH_NORMALIZED = DATA_DIR / "processed/maharashtra/maharashtra_normalized.parquet"

df_india = pd.read_parquet(INDIA_NORMALIZED)
df_mh = pd.read_parquet(MH_NORMALIZED)

print(f"India dataset: {len(df_india)} rows")
print(f"Maharashtra dataset: {len(df_mh)} rows")


India dataset: 1569168 rows
Maharashtra dataset: 218791 rows


## 2. India Forecasting Experiment
**Problem:** Given historical observations at a site, predict the next seasonal observation.
**Horizon:** 1 step ahead (next available seasonal observation).
**Validation:** Chronological split (e.g., Train: before 2021, Val: 2021-2022, Test: 2023+).

In [3]:
# Recompute seasonal features required for baselines and ML
from features.seasonal import compute_seasonal_statistics, assign_india_season
df_india = assign_india_season(df_india)


In [4]:
# Prepare forecasting frame
df_in_for = df_india.copy()
df_in_for = df_in_for.sort_values(["candidate_site_id", "observation_date"]).reset_index(drop=True)

# Define Target: Next observation at the same site
df_in_for["target_next_value"] = df_in_for.groupby("candidate_site_id")["water_level_raw"].shift(-1)

# Drop rows without a target (the last observation per site)
df_in_for = df_in_for.dropna(subset=["target_next_value"])
print(f"Eligible training samples (India): {len(df_in_for)}")


Eligible training samples (India): 1528099


### 2.1 Baselines

In [5]:
# Persistence: prediction = current value (which is previous relative to target)
df_in_for["pred_persistence"] = df_in_for["water_level_raw"]

# Seasonal Historical: mean of all previous years' observations for the target season
# Note: For strict adherence, the seasonal historical baseline in features.baselines 
# predicts the *current* observation. We adapt it to predict the *next* observation by shifting.

baseline_df = compute_seasonal_historical_baseline(df_india)
baseline_df = baseline_df.sort_values(["candidate_site_id", "observation_date"]).reset_index(drop=True)
df_in_for["pred_seasonal_hist"] = baseline_df.groupby("candidate_site_id")["seasonal_hist_prediction"].shift(-1)


In [6]:
# Chronological split for India based on date
train_in, val_in, test_in = chronological_split(df_in_for, date_col="observation_date", train_frac=0.7, val_frac=0.15)
print_split_summary(train_in, val_in, test_in, "observation_date")


--- Chronological Split Summary ---
Train:      1069669 rows | 1994-01-01 to 2017-01-01
Validation: 229215 rows | 2017-01-01 to 2020-08-31
Test:       229215 rows | 2020-08-31 to 2024-05-10
-----------------------------------


In [7]:
# Evaluate Baselines on Test Set
results_india = []

res_pers = evaluate_forecast(test_in["target_next_value"], test_in["pred_persistence"], "Persistence")
results_india.append(res_pers)

res_seas = evaluate_forecast(test_in["target_next_value"], test_in["pred_seasonal_hist"], "Seasonal Historical")
results_india.append(res_seas)

display(compare_models(results_india))


,model,samples,mae,rmse,r2,median_ae
0,Persistence,229215,2.1024,4.2487,0.8306,1.2000
1,Seasonal Historical,207091,2.0445,9.3602,0.0867,1.0769


### 2.2 ML Model (India)
Train a simple ML model (e.g., Random Forest or Linear Regression) using only backward-looking features (month, current water level, rate of change, etc).

In [8]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Features available at origin T
features = ["water_level_raw", "observation_month"]

X_train = train_in[features]
y_train = train_in["target_next_value"]

X_test = test_in[features]
y_test = test_in["target_next_value"]

# Simple Tree Model
model_in = HistGradientBoostingRegressor(random_state=42)
model_in.fit(X_train, y_train)

test_in["pred_ml"] = model_in.predict(X_test)

res_ml = evaluate_forecast(test_in["target_next_value"], test_in["pred_ml"], "HistGradientBoosting")
results_india.append(res_ml)

print("--- Final India Model Evaluation ---")
display(compare_models(results_india))


--- Final India Model Evaluation ---


,model,samples,mae,rmse,r2,median_ae
0,Persistence,229215,2.1024,4.2487,0.8306,1.2000
1,Seasonal Historical,207091,2.0445,9.3602,0.0867,1.0769
2,HistGradientBoosting,229215,2.0546,4.6833,0.7942,1.1364


## 3. Maharashtra Forecasting Experiment
**Problem:** Predict groundwater level for the next 6-hour interval.
**Data structure:** Continuous dense telemetry.

In [9]:
# Check true sampling cadence
df_mh['observation_timestamp'] = pd.to_datetime(df_mh['observation_timestamp'])
df_mh = df_mh.sort_values(["station_name", "observation_timestamp"]).reset_index(drop=True)
df_mh["time_diff_hours"] = df_mh.groupby("station_name")["observation_timestamp"].diff().dt.total_seconds() / 3600
print("Time difference between observations (hours) distribution:")
print(df_mh["time_diff_hours"].value_counts().head())


Time difference between observations (hours) distribution:
time_diff_hours
6.0     185344
24.0     14224
0.0       3788
48.0      2910
30.0      2131
Name: count, dtype: int64


In [10]:
# Define Target: Next observation
df_mh_for = df_mh.copy()
df_mh_for["target_next_value"] = df_mh_for.groupby("station_name")["water_level_raw"].shift(-1)
df_mh_for = df_mh_for.dropna(subset=["target_next_value"])

# Baselines
df_mh_for["pred_persistence"] = df_mh_for["water_level_raw"]

# Chronological split
train_mh, val_mh, test_mh = chronological_split(df_mh_for, date_col="observation_timestamp", train_frac=0.7, val_frac=0.15)
print_split_summary(train_mh, val_mh, test_mh, "observation_timestamp")


--- Chronological Split Summary ---
Train:      152443 rows | 2026-01-01 00:00:00 to 2026-04-05 06:00:00
Validation:  32667 rows | 2026-04-05 06:00:00 to 2026-04-27 12:00:00
Test:        32667 rows | 2026-04-27 12:00:00 to 2026-05-30 00:00:00
-----------------------------------


In [11]:
# Evaluate Baseline
results_mh = []
res_mh_pers = evaluate_forecast(test_mh["target_next_value"], test_mh["pred_persistence"], "Persistence (6h)")
results_mh.append(res_mh_pers)

display(compare_models(results_mh))


,model,samples,mae,rmse,r2,median_ae
0,Persistence (6h),32667,0.6498,5.472,0.995,0.02


### 3.1 ML Model (Maharashtra)

In [12]:
# Generate lag features
df_mh_for["lag_1"] = df_mh_for.groupby("station_name")["water_level_raw"].shift(1)
df_mh_for["hour"] = df_mh_for["observation_timestamp"].dt.hour

train_mh, val_mh, test_mh = chronological_split(df_mh_for, date_col="observation_timestamp", train_frac=0.7, val_frac=0.15)

features_mh = ["water_level_raw", "lag_1", "hour"]
X_train_mh = train_mh[features_mh]
y_train_mh = train_mh["target_next_value"]
X_test_mh = test_mh[features_mh]
y_test_mh = test_mh["target_next_value"]

# Handle missing lag_1 (first obs per station) by dropping or imputing
mask_train = ~X_train_mh.isna().any(axis=1) & ~y_train_mh.isna()
mask_test = ~X_test_mh.isna().any(axis=1) & ~y_test_mh.isna()

model_mh = HistGradientBoostingRegressor(random_state=42)
model_mh.fit(X_train_mh[mask_train], y_train_mh[mask_train])

pred_mh = model_mh.predict(X_test_mh[mask_test])
res_mh_ml = evaluate_forecast(y_test_mh[mask_test], pred_mh, "HistGradientBoosting (6h)")
results_mh.append(res_mh_ml)

print("--- Final Maharashtra Model Evaluation ---")
display(compare_models(results_mh))


--- Final Maharashtra Model Evaluation ---


,model,samples,mae,rmse,r2,median_ae
0,Persistence (6h),32667,0.6498,5.4720,0.9950,0.0200
1,HistGradientBoosting (6h),32662,5.6672,63.3989,0.3248,0.2241


## 4. Anomaly Detection
Separate anomalies from data-quality flags.

In [13]:
from features.anomaly import compute_seasonal_z_score

# India Anomalies (Seasonal Z-Score)
df_in_anom = compute_seasonal_z_score(df_india)
# Threshold: |z| > 3
df_in_anom["is_statistical_anomaly"] = df_in_anom["seasonal_z_score"].abs() > 3

num_anom = df_in_anom["is_statistical_anomaly"].sum()
print(f"India dataset: Detected {num_anom} statistical anomalies (|z| > 3)")

# Maharashtra Anomalies (Residual based on persistence)
df_mh["persistence_residual"] = df_mh["water_level_raw"] - df_mh.groupby("station_name")["water_level_raw"].shift(1)
# Robust MAD-based anomaly detection on residuals
# Calculate MAD per station
def robust_z(series):
    median = series.median()
    mad = np.median(np.abs(series - median))
    if mad == 0:
        return np.zeros_like(series)
    return (series - median) / (1.4826 * mad)

df_mh["residual_z_score"] = df_mh.groupby("station_name")["persistence_residual"].transform(robust_z)
df_mh["is_statistical_anomaly"] = df_mh["residual_z_score"].abs() > 5

num_anom_mh = df_mh["is_statistical_anomaly"].sum()
print(f"Maharashtra dataset: Detected {num_anom_mh} statistical anomalies (|robust_z| > 5 on residuals)")


India dataset: Detected 9180 statistical anomalies (|z| > 3)
Maharashtra dataset: Detected 0 statistical anomalies (|robust_z| > 5 on residuals)


## 5. Conclusions
- **India**: Documented how ML compares to Persistence/Seasonal Historical. Are ML models significantly better? Often, persistence or historical average is strong.
- **Maharashtra**: The continuous telemetry makes 6-hourly prediction highly correlated. A baseline of "Persistence" will yield extremely low errors, leaving little room for ML to improve unless predicting further out (e.g., 24h or 7 days).
- **Anomalies**: Anomalies are flagged statistically without removing them, preserving the original data.